# 04 — Model Evaluation

Notebook 04 owns **individual CNN validation evaluation only**: load selected checkpoints, generate deterministic validation predictions, calculate the three individual-model metrics, generate confusion matrices/ROC curves, and save the predictions with provenance for Notebook 05. It does not perform voting, optimize ensemble weights, freeze an ensemble or access the test partition.

All execution flags default to False. Use the actual private Kaggle checkpoints and original frozen manifest; committed experiment summaries contain no weights. No training, dataset downloads, splitting or experimental results occur by default. The reusable code remains in `src/glaucoma/`.

## 1. Kaggle / Colab / local paths and opt-ins
Use the same session, manifest, checkpoint bases, physical image-root mapping, evaluation name and inference settings in Notebook 05. Model construction/normalization/training are unchanged. The individual decision threshold is predefined at 0.5 and must be preserved by Notebook 05.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import copy
import json

RUN_PLATFORM_SETUP = False
CLONE_REPOSITORY = False
INSTALL_DEPENDENCIES = False
VERIFY_CHECKPOINTS = False
RUN_MODEL_VALIDATION = False
SHOW_SAVED_TABLES = False

PLATFORM = "kaggle" if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").is_dir()) else "colab" if Path("/content").is_dir() else "local"
default_repo = "/kaggle/working/glaucoma_detection" if PLATFORM == "kaggle" else "/content/glaucoma_detection" if PLATFORM == "colab" else str(Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent)
REPO_ROOT = Path(os.environ.get("PROJECT_REPO", default_repo))
REPO_URL = "https://github.com/VDJ2824/glaucoma_detection.git"
REPO_REF = "main"  # prefer a reviewed commit for reproducibility
BUNDLE_ROOT = Path("/kaggle/input/your-private-training-run")  # EDIT: run containing checkpoints/manifests/environment
if PLATFORM == "kaggle":
    default_session = BUNDLE_ROOT / "environment/session.json"
    default_output = Path("/kaggle/working/artifacts")
elif PLATFORM == "colab":
    default_output = Path("/content/drive/MyDrive/glaucoma_detection/artifacts")
    default_session = default_output / "runs/development/environment/session.json"
else:
    default_output = REPO_ROOT / "artifacts"
    default_session = default_output / "runs/development/environment/session.json"
SESSION_FILE = os.environ.get("GLAUCOMA_SESSION", str(default_session))
MANIFEST_CSV = BUNDLE_ROOT / "manifests/acrima_metadata.csv" if PLATFORM == "kaggle" else None
CHECKPOINT_ROOT = BUNDLE_ROOT / "checkpoints/core" if PLATFORM == "kaggle" else None
# Optional per-model bases containing <model>/<stage>/; leave None for a common base/session setting.
CHECKPOINT_ROOTS = None
# To decode original Kaggle manifest paths on another machine, set the CURRENT flat image directory.
# Keep the original CSV unchanged; this affects only in-memory physical path lookup.
FLAT_IMAGE_ROOT = None
OUTPUT_ROOT = default_output
EVALUATION_NAME = "acrima_phase4_v1"

INDIVIDUAL_THRESHOLD = 0.5  # predefined; Notebook 05 must retain this threshold


## 2. Independent environment setup
Kaggle is detected before Colab. Python 3.10–3.13 and the existing TensorFlow dependencies are supported. Mount Drive explicitly on Colab; attach your private training artifacts on Kaggle. Clone/install only when needed; installing stops execution for a kernel restart. No actual images or models are loaded by these default setup cells.

In [ ]:
if RUN_PLATFORM_SETUP:
    if not (3, 10) <= sys.version_info[:2] < (3, 14):
        raise RuntimeError("Use Python 3.10–3.13.")
    if PLATFORM == "colab":
        from google.colab import drive
        drive.mount("/content/drive")
    if CLONE_REPOSITORY:
        if REPO_ROOT.exists():
            raise FileExistsError("Existing clone preserved; inspect it rather than resetting it.")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT/"requirements-training.txt")], cwd=REPO_ROOT, check=True)
        raise RuntimeError("Restart the kernel, disable installation, then rerun.")
if not (REPO_ROOT/"src/glaucoma").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the existing project clone.")
sys.path.insert(0, str(REPO_ROOT/"src"))


## 3. Load saved session and checkpoint references
The original canonical manifest (including its original stored image paths) must match the training contracts. For physical relocation to another platform, use `FLAT_IMAGE_ROOT` for in-memory decoding rather than rewriting the CSV or contract hashes. Unknown/missing files, mismatched metadata and basename collisions fail explicitly.

In [ ]:
from glaucoma.utils.notebook_runtime import load_session
from glaucoma.ensemble.predictions import MEMBERS
from glaucoma.evaluation.checkpoints import verify_selected_checkpoints
from glaucoma.evaluation.workflow import run_directory, run_model_validation

cfg = copy.deepcopy(load_session(SESSION_FILE))
manifest_path = Path(MANIFEST_CSV or cfg["metadata"]["output"])
checkpoint_roots = CHECKPOINT_ROOTS or ({name: str(CHECKPOINT_ROOT) for name in MEMBERS} if CHECKPOINT_ROOT is not None else None)
selection_records = {
    "mobilenet_v2": REPO_ROOT/"docs/experiments/mobilenet_v2_kaggle_2026-11-26/metrics/selection.json",
    "efficientnet_b0": REPO_ROOT/"docs/experiments/efficientnet_b0_kaggle_2026-11-27/metrics/selection.json",
    "resnet50": REPO_ROOT/"docs/experiments/resnet50_kaggle_2026-11-27/metrics/selection.json",
}
directory = run_directory(cfg, EVALUATION_NAME, OUTPUT_ROOT)
print("Manifest:", manifest_path)
print("Evaluation artifacts:", directory)
print("Member order:", MEMBERS)


## 4. Optional checkpoint verification
Enable `VERIFY_CHECKPOINTS` to check both completed stages, saved states/contracts, selected weight-file hashes, frozen train/validation digest and the committed selection references. The actual loaded model name, head, backbone family/feature dimensions and graph are checked during inference. Neither this step nor validation uses test images or test labels for selection.

In [ ]:
if VERIFY_CHECKPOINTS:
    print(json.dumps(verify_selected_checkpoints(cfg, manifest_path, selection_records, checkpoint_roots), indent=2))
else:
    print("Checkpoint verification disabled.")


## 5. Individual-model validation inference and metrics
Enable `RUN_MODEL_VALIDATION` for a fresh evaluation directory. The shared function loads models sequentially, uses the unchanged model-specific preprocessing without augmentation, calls with training=False, preserves final partial batches, and aligns scores by dataset/image ID. It writes all three individual prediction CSVs plus an aligned table and a checksummed `validation_predictions.json` handoff.

Only individual metrics/plots are generated here: accuracy, precision, recall, specificity, F1, score-based ROC-AUC/BCE, three confusion matrices and three ROC curves. No voting or weight search runs, and no `frozen_ensemble.json` is produced. Model-level validation rows are selected from the saved manifest; test labels/pixels do not enter these calculations.

In [ ]:
model_validation_result = None
if RUN_MODEL_VALIDATION:
    model_validation_result = run_model_validation(cfg, manifest_path, enabled=True,
        name=EVALUATION_NAME, output_root=OUTPUT_ROOT, expected_records=selection_records,
        checkpoint_roots=checkpoint_roots, flat_image_root=FLAT_IMAGE_ROOT,
        threshold=INDIVIDUAL_THRESHOLD)
    print(json.dumps(model_validation_result, indent=2))
else:
    print("Individual-model validation disabled. No predictions generated.")


## 6. Inspect and hand off to Notebook 05
Optional read-only inspection of the saved individual metrics follows. Preserve the complete `phase4/<EVALUATION_NAME>/` directory. On Drive the same folder can be read from Notebook 05 directly. On Kaggle, save Notebook 04 outputs and attach them to Notebook 05; its opt-in `COPY_VALIDATION_HANDOFF` helper copies the verified pre-ensemble report directory into writable working storage, preserving all source bytes and hashes.

Notebook 05 must reuse these exact validation predictions and provenance, not regenerate them. All individual reports remain in `validation/`; Notebook 05's validation comparisons go in `validation/ensemble/`. Known-key/path/patient leakage fails, but image-level splitting without real patient identifiers still cannot establish patient independence. Single-dataset results are not external or clinical validation.

In [ ]:
if SHOW_SAVED_TABLES:
    import pandas as pd
    print(pd.read_csv(directory/"validation/metrics_comparison.csv").to_string(index=False))
    print(json.dumps(json.loads((directory/"validation_predictions.json").read_text()), indent=2))
else:
    print("Saved individual validation inspection disabled.")


## Next notebook
Continue to [05 — Ensemble Learning](05_ensemble_optimization.ipynb) for majority/equal/weighted voting, validation-only optimization, freezing and the separate final untouched-test opt-in. Do not access test images or tune anything on test outcomes here. See the [Phase 4 guide](../docs/phase4-ensemble-evaluation.md) for contracts, storage and limitations.